<a href="https://colab.research.google.com/github/purnimakushwaha/ITC101_Minor-project_python/blob/main/Quiz_generator_using_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# 🧠 QUIZ GENERATOR USING API
# Open Trivia Database API + JSON + ipywidgets
# ============================================================

import requests
import html
import random
import time
import os
import pandas as pd
import ipywidgets as widgets

from datetime import datetime
from IPython.display import display, HTML, clear_output


# ============================================================
# SETTINGS
# ============================================================

API_URL = "https://opentdb.com/api.php"

RESULT_FILE = "quiz_results.csv"

questions = []
current_question = 0
score = 0
correct_answers = 0
wrong_answers = 0
quiz_start_time = None
quiz_finished = False


# ============================================================
# LOAD PREVIOUS RESULTS
# ============================================================

if os.path.exists(RESULT_FILE):

    try:

        results_df = pd.read_csv(
            RESULT_FILE
        )

    except Exception:

        results_df = pd.DataFrame(
            columns=[
                "Date",
                "Category",
                "Difficulty",
                "Questions",
                "Score",
                "Percentage",
                "Time"
            ]
        )

else:

    results_df = pd.DataFrame(
        columns=[
            "Date",
            "Category",
            "Difficulty",
            "Questions",
            "Score",
            "Percentage",
            "Time"
        ]
    )


# ============================================================
# CATEGORY DATA
# ============================================================

categories = {
    "Any Category": "",
    "General Knowledge": "9",
    "Books": "10",
    "Film": "11",
    "Music": "12",
    "Television": "14",
    "Video Games": "15",
    "Science & Nature": "17",
    "Computers": "18",
    "Mathematics": "19",
    "Sports": "21",
    "Geography": "22",
    "History": "23",
    "Animals": "27"
}


# ============================================================
# DIFFICULTY
# ============================================================

difficulty_values = {
    "Any Difficulty": "",
    "Easy": "easy",
    "Medium": "medium",
    "Hard": "hard"
}


# ============================================================
# UI HEADER
# ============================================================

title = widgets.HTML(
    """
    <div style="
        background:#3949ab;
        color:white;
        padding:25px;
        border-radius:18px;
        text-align:center;
        font-family:Arial;
    ">

        <div style="
            font-size:32px;
            font-weight:bold;
        ">
            🧠 QUIZ GENERATOR
        </div>

        <div style="
            font-size:15px;
            margin-top:8px;


    </div>
    """
)


# ============================================================
# QUIZ SETTINGS
# ============================================================

category_dropdown = widgets.Dropdown(
    options=list(categories.keys()),
    value="Any Category",
    description="📚 Category:",
    layout=widgets.Layout(
        width="60%"
    )
)


difficulty_dropdown = widgets.Dropdown(
    options=list(difficulty_values.keys()),
    value="Any Difficulty",
    description="🎯 Difficulty:",
    layout=widgets.Layout(
        width="60%"
    )
)


question_count = widgets.BoundedIntText(
    value=5,
    min=1,
    max=50,
    step=1,
    description="🔢 Questions:",
    layout=widgets.Layout(
        width="60%"
    )
)


start_button = widgets.Button(
    description="🚀 Start Quiz",
    button_style="success",
    icon="play"
)


# ============================================================
# QUIZ AREA
# ============================================================

quiz_output = widgets.Output()


# ============================================================
# ANSWER BUTTONS
# ============================================================

answer_buttons = []


for i in range(4):

    button = widgets.Button(
        description="",
        layout=widgets.Layout(
            width="90%",
            height="45px",
            margin="5px"
        )
    )

    button.disabled = True

    answer_buttons.append(
        button
    )


# ============================================================
# NEXT BUTTON
# ============================================================

next_button = widgets.Button(
    description="➡️ Next Question",
    button_style="primary",
    disabled=True
)


# ============================================================
# RESULT / HISTORY BUTTONS
# ============================================================

history_button = widgets.Button(
    description="📊 Quiz History",
    button_style="info"
)


clear_history_button = widgets.Button(
    description="🗑️ Clear History",
    button_style="danger"
)


# ============================================================
# API FUNCTION
# ============================================================

def fetch_questions():

    global questions


    category = categories[
        category_dropdown.value
    ]


    difficulty = difficulty_values[
        difficulty_dropdown.value
    ]


    amount = question_count.value


    params = {

        "amount": amount,

        "type": "multiple"

    }


    if category:

        params["category"] = category


    if difficulty:

        params["difficulty"] = difficulty


    start_time = time.perf_counter()


    response = requests.get(
        API_URL,
        params=params,
        timeout=15
    )


    api_time = (
        time.perf_counter()
        - start_time
    )


    if response.status_code != 200:

        raise Exception(
            "API request failed."
        )


    data = response.json()


    if data["response_code"] != 0:

        raise Exception(
            "Could not generate the requested quiz."
        )


    questions = data["results"]


    return api_time


# ============================================================
# START QUIZ
# ============================================================

def start_quiz(button=None):

    global current_question
    global score
    global correct_answers
    global wrong_answers
    global quiz_start_time
    global quiz_finished


    try:

        start_button.disabled = True


        with quiz_output:

            clear_output()


            display(
                HTML(
                    """
                    <h3>
                    🔄 Generating questions from API...
                    </h3>
                    """
                )
            )


        api_time = fetch_questions()


        current_question = 0
        score = 0
        correct_answers = 0
        wrong_answers = 0
        quiz_finished = False

        quiz_start_time = time.time()


        for button in answer_buttons:

            button.disabled = False


        next_button.disabled = True


        show_question(
            api_time
        )


    except Exception as error:

        with quiz_output:

            clear_output()


            display(
                HTML(
                    f"""
                    <div style="
                        padding:20px;
                        background:#ffebee;
                        border-radius:12px;
                    ">

                    <h3>
                    ❌ Unable to Generate Quiz
                    </h3>

                    <p>
                    {html.escape(
                        str(error)
                    )}
                    </p>

                    <p>
                    Please check your internet
                    connection and try again.
                    </p>

                    </div>
                    """
                )
            )


    finally:

        start_button.disabled = False


# ============================================================
# SHOW QUESTION
# ============================================================

def show_question(
    api_time=None
):

    global current_question


    if current_question >= len(questions):

        finish_quiz()

        return


    question_data = questions[
        current_question
    ]


    question_text = html.unescape(
        question_data["question"]
    )


    correct_answer = html.unescape(
        question_data["correct_answer"]
    )


    incorrect_answers = [

        html.unescape(
            answer
        )

        for answer in
        question_data["incorrect_answers"]
    ]


    answers = incorrect_answers + [
        correct_answer
    ]


    random.shuffle(
        answers
    )


    # Store correct answer

    question_data[
        "correct_answer"
    ] = correct_answer


    question_data[
        "display_answers"
    ] = answers


    # --------------------------------------------------------
    # DISPLAY QUESTION
    # --------------------------------------------------------

    with quiz_output:

        clear_output()


        display(
            HTML(
                f"""
                <div style="
                    background:#f5f6fa;
                    padding:25px;
                    border-radius:18px;
                    margin-top:15px;
                    border:1px solid #dfe6e9;
                ">

                    <p style="
                        color:#636e72;
                    ">
                        Question
                        <b>
                        {current_question + 1}
                        </b>
                        of
                        <b>
                        {len(questions)}
                        </b>
                    </p>

                    <h2>
                        {html.escape(
                            question_text
                        )}
                    </h2>

                    <p>
                        📚
                        <b>
                        {html.unescape(
                            question_data["category"]
                        )}
                        </b>
                        &nbsp;&nbsp;

                        🎯
                        <b>
                        {question_data["difficulty"].title()}
                        </b>
                    </p>

                    <p>
                        🏆 Current Score:
                        <b>{score}</b>
                    </p>

                </div>
                """
            )
        )


    # --------------------------------------------------------
    # SET ANSWER BUTTONS
    # --------------------------------------------------------

    for i, button in enumerate(
        answer_buttons
    ):

        button.disabled = False
        button.button_style = ""
        button.description = answers[i]


    next_button.disabled = True


# ============================================================
# ANSWER CHECK
# ============================================================

def check_answer(
    button
):

    global score
    global correct_answers
    global wrong_answers


    question_data = questions[
        current_question
    ]


    selected_answer = button.description


    correct_answer = html.unescape(
        question_data["correct_answer"]
    )


    # Disable all buttons

    for btn in answer_buttons:

        btn.disabled = True


    if selected_answer == correct_answer:

        score += 1

        correct_answers += 1

        button.button_style = "success"


        message = """
        <h3 style="color:#2e7d32;">
            ✅ Correct Answer!
        </h3>
        """


    else:

        wrong_answers += 1

        button.button_style = "danger"


        # Highlight correct answer

        for btn in answer_buttons:

            if btn.description == correct_answer:

                btn.button_style = "success"


        message = f"""
        <h3 style="color:#c62828;">
            ❌ Wrong Answer
        </h3>

        <p>
            Correct answer:
            <b>
            {html.escape(correct_answer)}
            </b>
        </p>
        """


    with quiz_output:

        display(
            HTML(
                message
            )
        )


    next_button.disabled = False


# ============================================================
# NEXT QUESTION
# ============================================================

def next_question(
    button=None
):

    global current_question


    current_question += 1


    show_question()


# ============================================================
# FINISH QUIZ
# ============================================================

def finish_quiz():

    global quiz_finished


    quiz_finished = True


    total_questions = len(
        questions
    )


    percentage = (
        score /
        total_questions
    ) * 100


    total_time = (
        time.time()
        - quiz_start_time
    )


    if percentage >= 80:

        message = "🏆 Excellent!"

    elif percentage >= 60:

        message = "👏 Good Job!"

    elif percentage >= 40:

        message = "🙂 Keep Practicing!"

    else:

        message = "📚 More Practice Needed!"


    # --------------------------------------------------------
    # SAVE RESULT
    # --------------------------------------------------------

    save_quiz_result(
        percentage,
        total_time
    )


    with quiz_output:

        clear_output()


        display(
            HTML(
                f"""
                <div style="
                    background:#e8eaf6;
                    padding:30px;
                    border-radius:20px;
                    text-align:center;
                    border:2px solid #3949ab;
                ">

                    <div style="
                        font-size:55px;
                    ">
                        🏆
                    </div>

                    <h1>
                        QUIZ COMPLETED!
                    </h1>

                    <h2>
                        {message}
                    </h2>

                    <hr>

                    <h2>
                        Score:
                        {score} / {total_questions}
                    </h2>

                    <h2>
                        Percentage:
                        {percentage:.2f}%
                    </h2>

                    <p>
                        ✅ Correct:
                        <b>{correct_answers}</b>
                    </p>

                    <p>
                        ❌ Wrong:
                        <b>{wrong_answers}</b>
                    </p>

                    <p>
                        ⏱️ Time Taken:
                        <b>{total_time:.2f} seconds</b>
                    </p>

                    <hr>

                    <p>
                        Click
                        <b>
                        🚀 Start Quiz
                        </b>
                        to play again.
                    </p>

                </div>
                """
            )
        )


    for button in answer_buttons:

        button.disabled = True


    next_button.disabled = True


# ============================================================
# SAVE QUIZ RESULT
# ============================================================

def save_quiz_result(
    percentage,
    total_time
):

    global results_df


    new_result = {

        "Date":
            datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            ),

        "Category":
            category_dropdown.value,

        "Difficulty":
            difficulty_dropdown.value,

        "Questions":
            len(questions),

        "Score":
            score,

        "Percentage":
            round(
                percentage,
                2
            ),

        "Time":
            round(
                total_time,
                2
            )
    }


    results_df.loc[
        len(results_df)
    ] = new_result


    results_df.to_csv(
        RESULT_FILE,
        index=False
    )


# ============================================================
# SHOW HISTORY
# ============================================================

def show_history(
    button=None
):

    with quiz_output:

        clear_output()


        if results_df.empty:

            display(
                HTML(
                    """
                    <h3>
                    📊 No previous quiz results.
                    </h3>
                    """
                )
            )

            return


        display(
            HTML(
                """
                <h2>
                    📊 Quiz History
                </h2>
                """
            )
        )


        display(
            results_df
        )


# ============================================================
# CLEAR HISTORY
# ============================================================

def clear_history(
    button=None
):

    global results_df


    results_df = pd.DataFrame(
        columns=[
            "Date",
            "Category",
            "Difficulty",
            "Questions",
            "Score",
            "Percentage",
            "Time"
        ]
    )


    results_df.to_csv(
        RESULT_FILE,
        index=False
    )


    with quiz_output:

        clear_output()


        display(
            HTML(
                """
                <div style="
                    padding:20px;
                    background:#ffebee;
                    border-radius:12px;
                ">

                    <h3>
                        🗑️ Quiz history cleared.
                    </h3>

                </div>
                """
            )
        )


# ============================================================
# BUTTON EVENTS
# ============================================================

start_button.on_click(
    start_quiz
)


next_button.on_click(
    next_question
)


history_button.on_click(
    show_history
)


clear_history_button.on_click(
    clear_history
)


for button in answer_buttons:

    button.on_click(
        check_answer
    )


# ============================================================
# DISPLAY APP
# ============================================================

display(
    title
)


display(
    HTML(
        """
        <h3>
        ⚙️ Quiz Settings
        </h3>
        """
    )
)


display(
    category_dropdown
)


display(
    difficulty_dropdown
)


display(
    question_count
)


display(
    start_button
)


display(
    quiz_output
)


display(
    widgets.HBox(
        answer_buttons,
        layout=widgets.Layout(
            flex_direction="column",
            align_items="center"
        )
    )
)


display(
    next_button
)


display(
    widgets.HBox(
        [
            history_button,
            clear_history_button
        ],
        layout=widgets.Layout(
            justify_content="center",
            margin="20px"
        )
    )
)


print(
    "🧠 Quiz Generator started successfully!"
)

print(
    "Select settings and click Start Quiz."
)

HTML(value='\n    <div style="\n        background:#3949ab;\n        color:white;\n        padding:25px;\n    …

Dropdown(description='📚 Category:', layout=Layout(width='60%'), options=('Any Category', 'General Knowledge', …

Dropdown(description='🎯 Difficulty:', layout=Layout(width='60%'), options=('Any Difficulty', 'Easy', 'Medium',…

BoundedIntText(value=5, description='🔢 Questions:', layout=Layout(width='60%'), max=50, min=1)

Button(button_style='success', description='🚀 Start Quiz', icon='play', style=ButtonStyle())

Output()

Button(button_style='primary', description='➡️ Next Question', disabled=True, style=ButtonStyle())

🧠 Quiz Generator started successfully!
Select settings and click Start Quiz.
